# Fully Supervised with Linear Embeddings

## Prerequisites

In [ ]:
# install required libraries
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import numpy as np

In [ ]:
# checking if the GPU is enabled (saves time with training)
print(torch.cuda.is_available())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

False


In [ ]:
# using Google Drive to save checkpoints
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Transforms - augmentations


In [ ]:
"""
F.2 - the augmentations were:
- RandomHorizontalFlip
- RandomResizedCrop
- ColorJittering
- RandomGrayscaling
"""
transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomResizedCrop(size=32),
    transforms.RandomApply([transforms.ColorJitter(brightness=0.8, contrast=0.8, saturation=0.8, hue=0.2)], p=0.8),
    transforms.RandomGrayscale(p=0.2),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5),
                         (0.5, 0.5, 0.5))
])

embed_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5),
                         (0.5, 0.5, 0.5))
])

# SimCLR
For each image in a batch, SimCLR generates two different augmented views (positive pair)

In [ ]:
# generates two augmented views of the same image, for SimCLR contrastive learning
class SimCLRDataset(torch.utils.data.Dataset):
    def __init__(self, dataset, transform):
        self.dataset = dataset
        self.transform = transform

    def __getitem__(self, idx):
        x, _ = self.dataset[idx]
        x1 = self.transform(x)
        x2 = self.transform(x)
        return x1, x2

    def __len__(self):
        return len(self.dataset)

In [ ]:
# loading CIFAR-10 dataset, and wrapping it
trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True)
dataset = SimCLRDataset(trainset, transform)
loader = torch.utils.data.DataLoader(dataset, batch_size=512, shuffle=True, num_workers=4, drop_last=True) # every batch is of 512 size

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


# SimCLR Training Procedure

Sample a batch of unlabeled images.

Apply two random augmentations to each image to create positive pairs.

Pass both views through the encoder and projection head.

Calculate contrastive loss for each positive pair.

Backpropagate and update the encoder and projection head weights.

After training, discard the projection head and use the encoder for downstream tasks

In [ ]:
# using a pretrained resnet18 as implementation
resnet18 = torchvision.models.resnet18(weights=None, progress=True)
resnet18.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
resnet18.maxpool = nn.Identity()
resnet18.fc = nn.Identity()

# Projection Head

A small multilayer perceptron (MLP) that maps h to another representation
z

g
(
h
)
z=g(h), where contrastive loss is applied.
After training, the projection head is discarded only
f
(
x
)
f(x) is used for downstream tasks.

In [ ]:
# only used during training, discarded after
class ProjectionHead(nn.Module):
  def __init__(self):
    super().__init__()
    self.fc1 = nn.Linear(512, 512)
    self.bn = nn.BatchNorm1d(512)
    self.relu = nn.ReLU(inplace=True)
    self.fc2 = nn.Linear(512, 128)

  def forward(self, x):
    x = self.fc1(x)
    x = self.bn(x)
    x = self.relu(x)
    x = self.fc2(x)
    return F.normalize(x, dim=1)

In [ ]:
# SimCLR wrapping - passing both views through the encoder and the projection head
class SimCLR(nn.Module):
  def __init__(self):
    super().__init__()
    self.encoder = resnet18
    self.projection_head = ProjectionHead()

  def forward(self, x1, x2): # two images
      z1 = self.encoder(x1)
      z2 = self.encoder(x2)

      p1 = self.projection_head(z1)
      p2 = self.projection_head(z2)

      return p1, p2

  def embedding(self, x):
    return self.encoder(x)

## Contrastive Loss Function (NT-Xent Loss)
Normalized Temperature-scaled Cross Entropy Loss.

For a batch of N images, there are 2N samples (each with two views).

Each positive pair is contrasted against 2(N - 1) negative pairs.

Loss encourages similar pairs to have high cosine similarity:

In [ ]:
# contrastive loss
def nt_xent_loss(z1, z2, temperature=0.5):
    N = z1.size(0)

    z = torch.cat([z1, z2], dim=0)  # (2N, 128)

    similarity = F.cosine_similarity(z.unsqueeze(1), z.unsqueeze(0), dim=2)
    similarity /= temperature

    positives = torch.cat([
        torch.diag(similarity, N),
        torch.diag(similarity, -N)
    ], dim=0)  # (2N,)

    # mask self-similarity
    mask = torch.eye(2*N, dtype=torch.bool).to(z.device)
    similarity = similarity.masked_fill(mask, float('-inf'))

    logits = similarity  # (2N, 2N)
    labels = torch.cat([
        torch.arange(N, 2*N),  # first N samples match with last N
        torch.arange(0, N)     # last N samples match with first N
    ]).to(z.device)

    loss = F.cross_entropy(logits, labels)
    return loss

In [ ]:
model = SimCLR().to(device)

## Hyperparameters from SCAN
- SGD optimizer with 0.9 momentum
- Initial learning rate of 0.4 with a cosine scheduler
- Batch size was 512
- Weight decay of 0.0001

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.4, momentum=0.9, weight_decay=1e-4)
#T_max should be num_epochs
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=500, eta_min=0)

In [ ]:
dummy = torch.randn(2, 3, 32, 32).to(device)
h = model.encoder(dummy)
print(h.shape)  # should be torch.Size([2, 512])

torch.Size([2, 512])


Training model

In [ ]:
# for loading a checkpoint only..!

checkpoint_path = '/content/drive/MyDrive/5CCSAMLF/FS_Embed/models/simclr_latest.pth'
checkpoint = torch.load(checkpoint_path)
model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
start_epoch = checkpoint['epoch'] + 1

print(f"Loaded checkpoint from epoch {start_epoch}, loss: {checkpoint['loss']:.4f}")


RuntimeError: Attempting to deserialize object on a CUDA device but torch.cuda.is_available() is False. If you are running on a CPU-only machine, please use torch.load with map_location=torch.device('cpu') to map your storages to the CPU.

In [ ]:
checkpoint_path = '/content/drive/MyDrive/5CCSAMLF/FS_Embed/models/simclr_latest.pth'
start_epochs = 76
num_epochs = 500
best_loss = 1000 # initially set to a high number

for epoch in range(76, num_epochs): # or (start_epochs, num_epochs)
    model.train()
    running_loss = 0
    for (x1, x2) in loader:
        x1 = x1.to(device)
        x2 = x2.to(device)

        # Forward pass
        z1, z2 = model(x1, x2)

        loss = nt_xent_loss(z1, z2)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    scheduler.step()

    avg_loss = running_loss / len(loader)
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {avg_loss:.4f}")

    # save the best model (with the lowest loss)
    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save({'epoch': epoch,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'scheduler_state_dict': scheduler.state_dict(),
                    'loss': avg_loss},
                   checkpoint_path)
        print(f"Saved best model - epoch {epoch+1} with loss: {avg_loss:.4f}")

## Extracting embeddings

In [ ]:
# loading trained model once training has stopped
checkpoint_path = '/content/drive/MyDrive/5CCSAMLF/FS_Embed/models/simclr_latest.pth'
checkpoint = torch.load(checkpoint_path, map_location='cpu')

model = SimCLR().to(device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

In [ ]:
# 50k imgs
cifar_train = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=embed_transform)
cifar_trainloader = torch.utils.data.DataLoader(cifar_train, batch_size=512, shuffle=False)
# 10k imgs
cifar_test = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=embed_transform)
cifar_testloader = torch.utils.data.DataLoader(cifar_test, batch_size=512, shuffle=False)

In [ ]:
# train embeddings
train_embeddings = []
train_labels = []

with torch.no_grad():
    for imgs, labels in cifar_trainloader:
        imgs = imgs.to(device)
        embed = model.embedding(imgs)
        embed = F.normalize(embed, dim=1)
        train_embeddings.append(embed.cpu())
        train_labels.append(labels)
        print(f"Processed {len(train_embeddings) * 512} images")

train_embeddings_tensor = torch.cat(train_embeddings, dim=0)
train_labels_tensor = torch.cat(train_labels, dim=0)

In [ ]:
# save train embeddings
torch.save({
    'embeddings': train_embeddings_tensor,  # 50k imgs and batch size
    'labels': train_labels_tensor
}, '/content/drive/MyDrive/5CCSAMLF/FS_Embed/models/train_embeddings.pth')

In [ ]:
# test embeddings
test_embeddings = []
test_labels = []

with torch.no_grad():
    for imgs, labels in cifar_testloader:
        imgs = imgs.to(device)
        embed = model.embedding(imgs)
        embed = F.normalize(embed, dim=1)
        test_embeddings.append(embed.cpu())
        test_labels.append(labels)
        print(f"Processed {len(test_embeddings) * 512} images")

test_embeddings_tensor = torch.cat(test_embeddings, dim=0)
test_labels_tensor = torch.cat(test_labels, dim=0)     # ground truth

In [ ]:
# save test embeddings
torch.save({
    'embeddings': test_embeddings_tensor,  # (50000, 512) tensor
    'labels': test_labels_tensor           # (50000,) tensor
}, '/content/drive/MyDrive/5CCSAMLF/FS_Embed/models/test_embeddings.pth')